This code sets up the environment and ensures the user has the correct installations. 

In [1]:
"""Report the scientific Python dependencies used by this project."""

from __future__ import annotations

from importlib.metadata import PackageNotFoundError, version
import sys


_REQUIRED_DISTRIBUTIONS = (
    "qiskit",
    "qiskit-aer",
    "qiskit-nature",
    "qiskit-algorithms",
    "pyscf",
    "scipy",
    "matplotlib",
    "numpy",
)


def check_environment() -> dict[str, str]:
    """Return installed dependency versions or raise an actionable error."""
    if sys.version_info < (3, 10):
        raise RuntimeError("Python 3.10 or newer is required.")

    installed: dict[str, str] = {}
    missing: list[str] = []
    for distribution in _REQUIRED_DISTRIBUTIONS:
        try:
            installed[distribution] = version(distribution)
        except PackageNotFoundError:
            missing.append(distribution)

    if missing:
        raise RuntimeError(
            "Missing required packages: "
            + ", ".join(missing)
            + ". Install them with `python -m pip install -r requirements.txt`."
        )
    return installed


if __name__ == "__main__":
    for package, package_version in check_environment().items():
        print(f"{package}=={package_version}")


qiskit==2.6.2
qiskit-aer==0.18.2
qiskit-nature==0.8.0
qiskit-algorithms==0.4.0
pyscf==2.15.0
scipy==1.18.1
matplotlib==3.11.2
numpy==2.5.4


This sets up the problem with CHBrClF and reduces it, to be used later in the code. It also maps 

In [ ]:
"""Build a two-qubit active-space Hamiltonian for CHBrClF."""

from __future__ import annotations

from qiskit.quantum_info import SparsePauliOp
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.mappers import ParityMapper
from qiskit_nature.second_q.problems import ElectronicStructureProblem
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.units import DistanceUnit


def build_chbrclf_qubit_hamiltonian(
) -> tuple[
    ElectronicStructureProblem,
    ElectronicStructureProblem,
    ParityMapper,
    SparsePauliOp,
]:
    """Build and return the full and reduced CHBrClF electronic problems.

    The geometry is a tetrahedral Cartesian model in angstroms with distinct
    C-H, C-Br, C-Cl, and C-F bond lengths. The active-space transformer retains
    two electrons in two spatial orbitals and freezes the remaining orbitals.
    """
    atom = "\n".join(
        (
            "C  0.000000  0.000000  0.000000",
            "H  0.629312  0.629312  0.629312",
            "Br -1.120207 -1.120207  1.120207",
            "Cl -1.021911  1.021911 -1.021911",
            "F  0.779423 -0.779423 -0.779423",
        )
    )
    driver = PySCFDriver(
        atom=atom,
        basis="sto3g",
        charge=0,
        spin=0,
        unit=DistanceUnit.ANGSTROM,
    )
    electronic_problem = driver.run()

    transformer = ActiveSpaceTransformer(
        num_electrons=2,
        num_spatial_orbitals=2,
    )
    reduced_problem = transformer.transform(electronic_problem)
    mapper = ParityMapper(num_particles=reduced_problem.num_particles)
    qubit_hamiltonian = mapper.map(reduced_problem.hamiltonian.second_q_op())

    if qubit_hamiltonian.num_qubits != 2:
        raise RuntimeError(
            "Expected the active-space ParityMapper to produce a 2-qubit "
            f"Hamiltonian; got {qubit_hamiltonian.num_qubits} qubits."
        )

    return electronic_problem, reduced_problem, mapper, qubit_hamiltonian
